# OAI NDA Download Command Builder (dev)

Use this notebook to prepare NDA `downloadcmd` commands, not run downloads in Jupyter.

What this notebook does:
- pulls default paths from `OAI/.oai_env.json` (or `OAI/.oai_env.template.json`)
- populates package choices from `OAI/reference/oai_package_timepoint_map.csv` (or configured override)
- builds filtered S3 link files per package when metadata-driven filters are enabled
- writes a download plan JSON and prints a terminal `cmd` command for the tqdm runner


In [25]:
from __future__ import annotations

import json
import os
import time
import threading
import csv
import re
import shlex
import subprocess
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display

try:
    from tkinter import Tk, filedialog
except Exception:
    Tk = None
    filedialog = None

def find_repo_root(start: Path | None = None) -> Path:
    cursor = (start or Path.cwd()).resolve()
    for candidate in [cursor, *cursor.parents]:
        # Monorepo root (contains OAI/ module).
        if (candidate / 'AGENTS.md').exists() and (candidate / 'OAI').exists():
            return candidate
        # Standalone OAI repo root.
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'tmc_oai').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root for monorepo or standalone OAI checkout.')

def _resolve_path(path_text: str, base_dir: Path) -> Path:
    path = Path(path_text).expanduser()
    if not path.is_absolute():
        path = base_dir / path
    return path.resolve()

def load_oai_defaults(repo_root: Path) -> tuple[Path, Path, Path, Path]:
    oai_root = (repo_root / 'OAI').resolve() if (repo_root / 'OAI').exists() else repo_root.resolve()
    config_path = oai_root / '.oai_env.json'
    template_path = oai_root / '.oai_env.template.json'

    if config_path.exists():
        source = config_path
    elif template_path.exists():
        source = template_path
    else:
        raise FileNotFoundError('Could not find .oai_env.json or .oai_env.template.json under OAI root.')

    payload = json.loads(source.read_text(encoding='utf-8'))
    if not isinstance(payload, dict):
        raise ValueError(f'OAI config must be a JSON object: {source}')

    dataset_root_raw = str(payload.get('oai_dataset_root', '')).strip()
    default_dataset_root = _resolve_path(dataset_root_raw, source.parent) if dataset_root_raw else (oai_root / 'outputs').resolve()

    map_csv_raw = str(payload.get('timepoint_map_csv', '')).strip()
    map_csv = _resolve_path(map_csv_raw, source.parent) if map_csv_raw else (oai_root / 'reference' / 'oai_package_timepoint_map.csv').resolve()

    return source, oai_root, default_dataset_root, map_csv


def load_package_options(map_csv: Path) -> tuple[list[tuple[str, str]], pd.DataFrame]:
    frame = pd.read_csv(map_csv, dtype=str).fillna('')
    if 'package_number' not in frame.columns:
        raise KeyError(f"Expected 'package_number' column in {map_csv}")

    for column in ('package_number', 'timepoint_label', 'notes'):
        if column not in frame.columns:
            frame[column] = ''

    frame = frame[['package_number', 'timepoint_label', 'notes']].copy()
    frame['package_number'] = frame['package_number'].astype(str).str.strip()
    frame['timepoint_label'] = frame['timepoint_label'].astype(str).str.strip()
    frame['notes'] = frame['notes'].astype(str).str.strip()
    frame = frame[frame['package_number'].str.fullmatch(r'\d+')]
    frame = frame.drop_duplicates(subset=['package_number'], keep='first')
    frame = frame.sort_values('package_number', key=lambda s: s.astype(int)).reset_index(drop=True)

    options: list[tuple[str, str]] = []
    for row in frame.itertuples(index=False):
        label_parts = [f'Package {row.package_number}']
        if row.timepoint_label:
            label_parts.append(f'timepoint={row.timepoint_label}')
        if row.notes:
            label_parts.append(f'notes={row.notes}')
        options.append((' | '.join(label_parts), row.package_number))
    return options, frame


In [26]:
repo_root = find_repo_root()
config_source, oai_root, default_download_root, default_map_csv = load_oai_defaults(repo_root)

display(Markdown(
    f'**Repo root:** `{repo_root}`  \n'
    f'**OAI config source:** `{config_source}`  \n'
    f'**Default download root:** `{default_download_root}`  \n'
    f'**Default package map CSV:** `{default_map_csv}`'
))


**Repo root:** `D:\TMCProject_V2`  
**OAI config source:** `D:\TMCProject_V2\OAI\.oai_env.json`  
**Default download root:** `E:\OAI`  
**Default package map CSV:** `D:\TMCProject_V2\OAI\reference\oai_package_timepoint_map.csv`

In [27]:
import os
import textwrap

auth_username = widgets.Text(
    value='',
    description='NDA user',
    placeholder='your_nda_username',
    layout=widgets.Layout(width='420px'),
)
auth_password = widgets.Password(
    value='',
    description='NDA pass',
    placeholder='NDA account password',
    layout=widgets.Layout(width='420px'),
)
validate_auth = widgets.Checkbox(value=True, description='Validate credentials now')
save_auth_button = widgets.Button(description='Save NDA auth', button_style='warning')
auth_status = widgets.HTML(value='')
auth_output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', max_height='220px', overflow='auto'))


def save_nda_auth(_: widgets.Button) -> None:
    username = auth_username.value.strip().lower()
    password = auth_password.value
    with auth_output:
        clear_output(wait=True)
        if not username:
            auth_status.value = '<b style="color:#a94442;">Enter your NDA username.</b>'
            print('Username is required.')
            return
        if not password:
            auth_status.value = '<b style="color:#a94442;">Enter your NDA password.</b>'
            print('Password is required.')
            return

        script = textwrap.dedent(
            '''
import configparser
import os
from pathlib import Path

import NDATools
import keyring
from NDATools.upload.submission.api import UserApi

username = os.environ['NDA_USERNAME_INPUT'].strip().lower()
password = os.environ['NDA_PASSWORD_INPUT']
validate_now = os.environ.get('NDA_VALIDATE_AUTH', '1') == '1'

NDATools.prerun_checks_and_setup()
cfg_path = Path(NDATools.NDA_TOOLS_SETTINGS_CFG_FILE)
cfg = configparser.ConfigParser()
cfg.read(cfg_path, encoding='utf-8')
if not cfg.has_section('User'):
    cfg.add_section('User')
cfg.set('User', 'username', username)
with cfg_path.open('w', encoding='utf-8') as handle:
    cfg.write(handle)

keyring.set_password(NDATools.SERVICE_NAME, username, password)
print(f'Saved username to: {cfg_path}')
print('Saved password to OS keyring for service: nda-tools')

if validate_now:
    endpoint = cfg.get('Endpoints', 'user', fallback='https://nda.nih.gov/api/user')
    valid = UserApi(endpoint).is_valid_nda_credentials(username, password)
    print(f'Credential check against {endpoint}: {valid}')
    if not valid:
        raise SystemExit(2)
            '''
        )

        env = os.environ.copy()
        env['NDA_USERNAME_INPUT'] = username
        env['NDA_PASSWORD_INPUT'] = password
        env['NDA_VALIDATE_AUTH'] = '1' if validate_auth.value else '0'

        proc = subprocess.run(
            ['uv', 'run', '--with', 'nda-tools', 'python', '-'],
            input=script,
            text=True,
            capture_output=True,
            check=False,
            cwd=repo_root,
            env=env,
        )

        if proc.stdout.strip():
            print(proc.stdout.strip())
        if proc.stderr.strip():
            print(proc.stderr.strip())

        auth_password.value = ''
        if proc.returncode == 0:
            auth_status.value = '<b style="color:#2f6f3e;">NDA credentials saved. Downloads should run without interactive credential prompts.</b>'
        else:
            auth_status.value = '<b style="color:#a94442;">Could not validate/save NDA credentials. Check output below.</b>'


save_auth_button.on_click(save_nda_auth)

display(widgets.VBox([
    widgets.HTML('<b>One-time NDA auth setup</b>: save username to <code>~/.NDATools/settings.cfg</code> and password to OS keyring.'),
    widgets.HBox([auth_username, auth_password]),
    widgets.HBox([validate_auth, save_auth_button]),
    auth_status,
    auth_output,
]))

In [28]:
from IPython.display import Javascript

map_csv_text = widgets.Text(
    value=str(default_map_csv),
    description='Map CSV',
    layout=widgets.Layout(width='900px'),
)
reload_packages_button = widgets.Button(description='Reload packages', button_style='info')
filter_text = widgets.Text(
    value='',
    description='Filter',
    placeholder='Filter by package id or timepoint label',
    layout=widgets.Layout(width='900px'),
)
package_selector = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Packages',
    rows=12,
    layout=widgets.Layout(width='900px'),
)
select_all_button = widgets.Button(description='Select all')
clear_selection_button = widgets.Button(description='Clear selection')
manual_ids_text = widgets.Textarea(
    value='',
    description='Manual IDs',
    placeholder='Optional: comma/newline separated package IDs',
    rows=3,
    layout=widgets.Layout(width='900px'),
)
download_dir_text = widgets.Text(
    value=str(default_download_root),
    description='Folder',
    layout=widgets.Layout(width='760px'),
)
browse_button = widgets.Button(description='Browse folder')
username_text = widgets.Text(
    value='',
    description='NDA user',
    placeholder='Optional (downloadcmd prompts if needed)',
    layout=widgets.Layout(width='450px'),
)
datastructure_text = widgets.Text(
    value='',
    description='Data struct',
    placeholder='Optional: e.g., image03',
    layout=widgets.Layout(width='320px'),
)
file_regex_text = widgets.Text(
    value='',
    description='Regex',
    placeholder='Optional: e.g., (?i).*\\.(jpg|tar\\.gz)$',
    layout=widgets.Layout(width='580px'),
)
metadata_contains_text = widgets.Text(
    value='',
    description='Meta text',
    placeholder='Optional: contains text in metadata (e.g., xray)',
    layout=widgets.Layout(width='620px'),
)
image_type_select = widgets.SelectMultiple(
    options=[
        ('MRI', 'mri'),
        ('X-Ray', 'x-ray'),
        ('Other', 'other'),
    ],
    value=('x-ray', 'other'),
    description='Image type',
    rows=3,
    layout=widgets.Layout(width='260px'),
)
joint_filter_select = widgets.SelectMultiple(
    options=[
        ('Knee', 'knee'),
        ('Hip', 'hip'),
        ('Hand', 'hand'),
    ],
    value=('knee', 'hip', 'hand'),
    description='Joint',
    rows=3,
    layout=widgets.Layout(width='240px'),
)
extension_filter_select = widgets.SelectMultiple(
    options=[
        ('JPG (.jpg)', '.jpg'),
        ('TAR.GZ (.tar.gz)', '.tar.gz'),
        ('PNG (.png)', '.png'),
        ('DICOM (.dcm)', '.dcm'),
        ('NIfTI (.nii.gz)', '.nii.gz'),
    ],
    value=(),
    description='Ext',
    rows=5,
    layout=widgets.Layout(width='280px'),
)
worker_threads = widgets.IntSlider(
    value=32,
    min=1,
    max=64,
    step=1,
    description='Threads',
    readout=True,
    layout=widgets.Layout(width='300px'),
)
verify_only = widgets.Checkbox(value=False, description='Verify-only (no download)')
verbose_logs = widgets.Checkbox(value=False, description='Verbose logs')
auto_install_downloader = widgets.Checkbox(value=True, description='Auto-install downloader env')
run_button = widgets.Button(description='Build commands', button_style='success')
copy_runner_cmd_button = widgets.Button(description='Copy runner cmd', button_style='info', disabled=True)
install_downloader_button = widgets.Button(description='Install downloader', button_style='info')
pause_button = widgets.Button(description='Pause current', button_style='warning', disabled=True)
cancel_button = widgets.Button(description='Cancel run', button_style='danger', disabled=True)
run_button.layout = widgets.Layout(width='140px', min_width='140px', flex='0 0 auto')
copy_runner_cmd_button.layout = widgets.Layout(width='150px', min_width='150px', flex='0 0 auto')
install_downloader_button.layout = widgets.Layout(width='160px', min_width='160px', flex='0 0 auto')
pause_button.layout = widgets.Layout(width='130px', min_width='130px', flex='0 0 auto')
cancel_button.layout = widgets.Layout(width='120px', min_width='120px', flex='0 0 auto')
status_html = widgets.HTML(value='')
current_package_html = widgets.HTML(value='<b>Current package:</b> idle')
package_files_html = widgets.HTML(value='<b>Package files:</b> 0 / ?')
package_file_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description='Files',
    bar_style='',
    layout=widgets.Layout(width='520px'),
)
package_file_meta_html = widgets.HTML(
    value='<span style="white-space:nowrap;">0.0% | ETA --</span>',
    layout=widgets.Layout(width='260px'),
)
overall_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description='Packages',
    bar_style='',
    layout=widgets.Layout(width='520px'),
)
overall_progress_meta_html = widgets.HTML(
    value='<span style="white-space:nowrap;">0.0% | ETA --</span>',
    layout=widgets.Layout(width='260px'),
)
network_usage_html = widgets.HTML(
    value='<b>Network:</b> --',
    layout=widgets.Layout(width='520px'),
)
output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', max_height='450px', overflow='auto'))
log_scroll_class = 'oai-download-log-output'
log_scroll_state_key = f'__tmc_log_scroll_state__{log_scroll_class}'
output_area.add_class(log_scroll_class)

state: dict[str, object] = {
    'all_options': [],
    'last_runner_cmd': '',
    'last_install_cmd': '',
    'downloader_env_ready': False,
}
runtime_state: dict[str, object] = {
    'is_running': False,
    'pause_requested': False,
    'cancel_requested': False,
    'active_process': None,
    'worker_thread': None,
    'auto_pause_detected': False,
    'auto_resume_enabled': True,
    'auto_resume_thread': None,
    'auto_resume_stop_event': None,
    'resume_payload': None,
}
runtime_lock = threading.Lock()
PROGRESS_UPDATE_INTERVAL_SECONDS = 60.0
NETWORK_UPDATE_INTERVAL_SECONDS = 10.0
MONITOR_SCAN_INTERVAL_SECONDS = 10.0
AUTO_RESUME_INTERVAL_SECONDS = 60.0
DNS_RETRY_MARKERS = (
    'nameresolutionerror',
    'failed to resolve',
    'getaddrinfo failed',
    'temporary failure in name resolution',
)
progress_update_state: dict[str, float] = {}


def parse_manual_package_ids(raw_text: str) -> list[str]:
    tokens = re.findall(r'\d+', raw_text)
    seen: set[str] = set()
    ordered: list[str] = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            ordered.append(token)
    return ordered


def _should_emit_progress_update(
    key: str,
    force: bool = False,
    interval_seconds: float | None = None,
) -> bool:
    threshold = PROGRESS_UPDATE_INTERVAL_SECONDS if interval_seconds is None else max(0.0, float(interval_seconds))
    now = time.monotonic()
    with runtime_lock:
        if force:
            progress_update_state[key] = now
            return True
        previous = float(progress_update_state.get(key, 0.0))
        if now - previous >= threshold:
            progress_update_state[key] = now
            return True
    return False


def _sync_log_scroll_position() -> None:
    script = f"""
(function() {{
  const className = '{log_scroll_class}';
  const stateKey = '{log_scroll_state_key}';
  const roots = document.getElementsByClassName(className);
  if (!roots || roots.length === 0) {{
    return;
  }}

  const root = roots[roots.length - 1];
  const scrollEl =
    root.querySelector('.jupyter-widgets-output-area') ||
    root.querySelector('.output_scroll') ||
    root.querySelector('.output_area') ||
    root;
  if (!scrollEl) {{
    return;
  }}

  const state = window[stateKey] || {{ nearBottom: true, scrollTop: 0 }};
  const updateState = () => {{
    const distance = scrollEl.scrollHeight - (scrollEl.scrollTop + scrollEl.clientHeight);
    state.nearBottom = distance <= Math.max(12, scrollEl.clientHeight * 0.10);
    state.scrollTop = scrollEl.scrollTop;
    window[stateKey] = state;
  }};

  if (!scrollEl.dataset.tmcLogScrollBound) {{
    scrollEl.addEventListener('scroll', updateState, {{ passive: true }});
    scrollEl.dataset.tmcLogScrollBound = '1';
  }}

  if (state.nearBottom) {{
    scrollEl.scrollTop = scrollEl.scrollHeight;
  }} else {{
    const maxTop = Math.max(0, scrollEl.scrollHeight - scrollEl.clientHeight);
    const priorTop = Number.isFinite(state.scrollTop) ? state.scrollTop : scrollEl.scrollTop;
    scrollEl.scrollTop = Math.min(maxTop, Math.max(0, priorTop));
  }}

  updateState();
}})();
"""
    display(Javascript(script))


def append_log(message: str) -> None:
    text = str(message)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    if not text.endswith('\n'):
        text += '\n'
    output_area.append_stdout(text)


def set_controls_running(is_running: bool) -> None:
    run_button.disabled = is_running
    pause_button.disabled = not is_running
    cancel_button.disabled = not is_running


def is_pause_requested() -> bool:
    with runtime_lock:
        return bool(runtime_state.get('pause_requested', False))


def is_cancel_requested() -> bool:
    with runtime_lock:
        return bool(runtime_state.get('cancel_requested', False))


def line_has_dns_resolution_error(line: str) -> bool:
    text = str(line or '').strip().lower()
    if not text:
        return False
    if 's3.amazonaws.com' not in text:
        return False
    return any(marker in text for marker in DNS_RETRY_MARKERS)


def ensure_resume_payload(
    package_ids: list[str],
    download_dir: Path,
    download_filters: dict[str, object],
) -> None:
    payload = {
        'package_ids': [str(value).strip() for value in package_ids if str(value).strip()],
        'download_dir': str(download_dir),
        'download_filters': dict(download_filters),
    }
    with runtime_lock:
        runtime_state['resume_payload'] = payload


def clear_resume_payload() -> None:
    with runtime_lock:
        runtime_state['resume_payload'] = None


def stop_auto_resume_timer() -> None:
    with runtime_lock:
        stop_event = runtime_state.get('auto_resume_stop_event')
        runtime_state['auto_resume_stop_event'] = None
        runtime_state['auto_resume_thread'] = None
    if isinstance(stop_event, threading.Event):
        stop_event.set()


def start_download_worker(
    package_ids: list[str],
    download_dir: Path,
    download_filters: dict[str, object],
    *,
    status_message: str,
    is_auto_resume: bool = False,
) -> bool:
    with runtime_lock:
        worker = runtime_state.get('worker_thread')
        already_running = isinstance(worker, threading.Thread) and worker.is_alive()
        if already_running:
            return False
        runtime_state['is_running'] = True
        runtime_state['pause_requested'] = False
        runtime_state['cancel_requested'] = False
        runtime_state['auto_pause_detected'] = False

    if not is_auto_resume:
        stop_auto_resume_timer()

    ensure_resume_payload(package_ids, download_dir, download_filters)

    set_controls_running(True)
    status_html.value = status_message

    worker_thread = threading.Thread(
        target=run_downloads_worker,
        args=(package_ids, download_dir, download_filters),
        daemon=True,
    )
    with runtime_lock:
        runtime_state['worker_thread'] = worker_thread
    worker_thread.start()
    return True


def _auto_resume_loop(stop_event: threading.Event) -> None:
    try:
        while not stop_event.wait(AUTO_RESUME_INTERVAL_SECONDS):
            with runtime_lock:
                auto_resume_enabled = bool(runtime_state.get('auto_resume_enabled', True))
                auto_pause_detected = bool(runtime_state.get('auto_pause_detected', False))
                worker = runtime_state.get('worker_thread')
                is_running = bool(runtime_state.get('is_running', False))
                payload = runtime_state.get('resume_payload')

            if not auto_resume_enabled or not auto_pause_detected:
                return

            if is_running or (isinstance(worker, threading.Thread) and worker.is_alive()):
                continue

            if not isinstance(payload, dict):
                append_log('Auto-resume skipped: missing resume payload.')
                continue

            package_ids = [str(value).strip() for value in payload.get('package_ids', []) if str(value).strip()]
            download_dir_text = str(payload.get('download_dir') or '').strip()
            filters_raw = payload.get('download_filters')
            download_filters = dict(filters_raw) if isinstance(filters_raw, dict) else {}

            if not package_ids or not download_dir_text:
                append_log('Auto-resume skipped: incomplete resume payload.')
                continue

            download_dir = Path(download_dir_text).expanduser().resolve()
            download_dir.mkdir(parents=True, exist_ok=True)

            append_log('Auto-resume timer fired (60s). Attempting to resume downloads...')
            started = start_download_worker(
                package_ids,
                download_dir,
                download_filters,
                status_message='<b style="color:#2f6f3e;">Auto-resume retry started after DNS issue.</b>',
                is_auto_resume=True,
            )
            if started:
                return
    finally:
        with runtime_lock:
            if runtime_state.get('auto_resume_stop_event') is stop_event:
                runtime_state['auto_resume_stop_event'] = None
                runtime_state['auto_resume_thread'] = None


def start_auto_resume_timer() -> None:
    with runtime_lock:
        if not bool(runtime_state.get('auto_resume_enabled', True)):
            return
        existing_thread = runtime_state.get('auto_resume_thread')
        if isinstance(existing_thread, threading.Thread) and existing_thread.is_alive():
            return
        stop_event = threading.Event()
        timer_thread = threading.Thread(
            target=_auto_resume_loop,
            args=(stop_event,),
            daemon=True,
        )
        runtime_state['auto_resume_stop_event'] = stop_event
        runtime_state['auto_resume_thread'] = timer_thread
    timer_thread.start()


def trigger_auto_pause_due_to_network(line: str) -> None:
    with runtime_lock:
        if not bool(runtime_state.get('is_running', False)):
            return
        if bool(runtime_state.get('auto_pause_detected', False)):
            return
        runtime_state['auto_pause_detected'] = True
        runtime_state['pause_requested'] = True
        active_process = runtime_state.get('active_process')

    status_html.value = '<b style="color:#8a6d3b;">DNS resolution issue detected. Pausing now; auto-resume will retry every 1 minute.</b>'
    current_package_html.value = '<b>Current package:</b> auto-pause requested'
    append_log('Auto-pause requested: DNS resolution failure while accessing s3.amazonaws.com.')
    append_log(f'Network error line: {line}')

    terminate_process_tree(active_process)


def terminate_process_tree(process) -> None:
    if process is None:
        return
    try:
        if process.poll() is not None:
            return
    except Exception:
        return

    try:
        if os.name == 'nt':
            subprocess.run(
                ['taskkill', '/PID', str(process.pid), '/T', '/F'],
                capture_output=True,
                text=True,
                check=False,
            )
        else:
            process.terminate()
    except Exception:
        try:
            process.terminate()
        except Exception:
            pass


def apply_filter() -> None:
    all_options = state.get('all_options', [])
    query = filter_text.value.strip().lower()
    if not query:
        package_selector.options = all_options
        return
    filtered = [
        item
        for item in all_options
        if query in item[0].lower() or query in item[1].lower()
    ]
    package_selector.options = filtered


def refresh_package_options(_: widgets.Button | None = None) -> None:
    map_csv = Path(map_csv_text.value).expanduser().resolve()
    with output_area:
        clear_output(wait=True)
        if not map_csv.exists():
            print(f'Package map CSV not found: {map_csv}')
            state['all_options'] = []
            package_selector.options = []
            status_html.value = '<b style="color:#a94442;">Package map CSV not found.</b>'
            return
        try:
            options, frame = load_package_options(map_csv)
        except Exception as exc:
            print(f'Failed to parse package map CSV: {exc}')
            state['all_options'] = []
            package_selector.options = []
            status_html.value = '<b style="color:#a94442;">Failed to parse package map CSV.</b>'
            return
        state['all_options'] = options
        apply_filter()
        print(f'Loaded {len(options)} package options from {map_csv}')
        display(frame.head(20))
        status_html.value = f'<b style="color:#2f6f3e;">Loaded {len(options)} package options.</b>'


def on_filter_change(change: dict) -> None:
    if change.get('name') == 'value':
        apply_filter()


def on_select_all(_: widgets.Button) -> None:
    package_selector.value = tuple(value for _, value in package_selector.options)


def on_clear_selection(_: widgets.Button) -> None:
    package_selector.value = ()


def choose_folder(_: widgets.Button) -> None:
    with output_area:
        clear_output(wait=True)
        if Tk is None or filedialog is None:
            print('tkinter file dialog is unavailable in this Python environment.')
            return
        root = Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        selected = filedialog.askdirectory(initialdir=download_dir_text.value or str(default_download_root))
        root.destroy()
        if selected:
            download_dir_text.value = selected
            print(f'Selected folder: {selected}')
        else:
            print('Folder selection canceled.')


def _copy_to_clipboard(text: str) -> tuple[bool, str]:
    payload = str(text or '')
    if not payload.strip():
        return False, 'No runner command available to copy yet.'

    ps_error = ''
    try:
        proc = subprocess.run(
            ['powershell', '-NoProfile', '-Command', 'Set-Clipboard -Value $input'],
            input=payload,
            text=True,
            capture_output=True,
            check=False,
        )
        if proc.returncode == 0:
            return True, 'Runner command copied to clipboard.'
        ps_error = (proc.stderr or proc.stdout or f'exit {proc.returncode}').strip()
    except Exception as exc:
        ps_error = str(exc)

    if Tk is not None:
        try:
            root = Tk()
            root.withdraw()
            root.clipboard_clear()
            root.clipboard_append(payload)
            root.update()
            root.destroy()
            return True, 'Runner command copied to clipboard.'
        except Exception as exc:
            tk_error = str(exc)
            detail = f'PowerShell clipboard failed: {ps_error}; Tk fallback failed: {tk_error}'
            return False, detail

    return False, f'Clipboard copy failed: {ps_error}'


def _cmd_join(parts: list[str]) -> str:
    return subprocess.list2cmdline([str(part) for part in parts])


def ensure_downloader_environment(*, verbose: bool = True) -> bool:
    sync_cmd_parts = ['uv', 'sync', '--project', str(oai_root)]
    bootstrap_cmd_parts = [
        'uv',
        'run',
        '--project',
        str(oai_root),
        '--with',
        'tqdm',
        'python',
        '-c',
        'import tmc_oai; import tqdm; print("tmc_oai downloader env ready")',
    ]

    sync_cmd = _cmd_join(sync_cmd_parts)
    bootstrap_cmd = _cmd_join(bootstrap_cmd_parts)
    state['last_install_cmd'] = sync_cmd + ' && ' + bootstrap_cmd

    if verbose:
        append_log('Ensuring OAI downloader package is installed/updated (Windows CMD commands):')
        append_log('CMD: ' + sync_cmd)
        append_log('CMD: ' + bootstrap_cmd)

    sync_proc = subprocess.run(
        sync_cmd_parts,
        cwd=oai_root,
        text=True,
        capture_output=True,
        check=False,
    )

    sync_out = (sync_proc.stdout or '').strip()
    sync_err = (sync_proc.stderr or '').strip()
    if sync_out:
        append_log(sync_out)
    if sync_err:
        append_log(sync_err)
    if sync_proc.returncode != 0:
        state['downloader_env_ready'] = False
        append_log(f'Downloader install/update failed during sync (exit {sync_proc.returncode}).')
        return False

    bootstrap_proc = subprocess.run(
        bootstrap_cmd_parts,
        cwd=oai_root,
        text=True,
        capture_output=True,
        check=False,
    )

    boot_out = (bootstrap_proc.stdout or '').strip()
    boot_err = (bootstrap_proc.stderr or '').strip()
    if boot_out:
        append_log(boot_out)
    if boot_err:
        append_log(boot_err)

    if bootstrap_proc.returncode == 0:
        state['downloader_env_ready'] = True
        return True

    state['downloader_env_ready'] = False
    append_log(f'Downloader bootstrap check failed (exit {bootstrap_proc.returncode}).')
    return False


def install_downloader(_: widgets.Button) -> None:
    with output_area:
        clear_output(wait=True)

    ok = ensure_downloader_environment(verbose=True)
    if ok:
        status_html.value = '<b style="color:#2f6f3e;">Downloader environment is ready.</b>'
    else:
        status_html.value = '<b style="color:#a94442;">Failed to install/update downloader environment. See output.</b>'

    _sync_log_scroll_position()


def copy_runner_command(_: widgets.Button) -> None:
    runner_cmd = str(state.get('last_runner_cmd') or '').strip()
    ok, message = _copy_to_clipboard(runner_cmd)
    if ok:
        status_html.value = '<b style="color:#2f6f3e;">Runner command copied to clipboard.</b>'
        append_log('Copied runner command to clipboard.')
    else:
        status_html.value = '<b style="color:#a94442;">Could not copy runner command. See output.</b>'
        append_log(message)


def _normalize_asset_id_token(raw_value: str | None) -> str:
    text = str(raw_value or '').strip().lower()
    if not text:
        return ''
    leaf = text.rsplit('/', maxsplit=1)[-1]
    for suffix in ('.tar.gz', '.nii.gz', '.jpg', '.jpeg', '.dcm'):
        if leaf.endswith(suffix):
            leaf = leaf[: -len(suffix)]
            break
    leaf = leaf.removesuffix('_1x1').removesuffix('_2x2')
    return leaf.strip()


def _classify_joint_from_description(description: str | None) -> str:
    text = str(description or '').strip().lower()
    if 'hand' in text:
        return 'hand'
    if 'knee' in text:
        return 'knee'
    if 'hip' in text or 'pelvis' in text:
        return 'hip'
    return ''


def _classify_image_type(modality: str | None, description: str | None) -> str:
    modality_text = str(modality or '').strip().lower()
    description_text = str(description or '').strip().lower()
    merged = f'{modality_text} {description_text}'
    if modality_text == 'x-ray' or 'xray' in merged or 'x-ray' in merged:
        return 'x-ray'
    if 'mri' in merged or re.search(r'\\bmr\\b', merged):
        return 'mri'
    if _classify_joint_from_description(description_text):
        return 'x-ray'
    return 'other'


def _read_table_with_fallback(path: Path) -> pd.DataFrame:
    encodings = ('utf-8-sig', 'utf-16', 'utf-16-le', 'utf-16-be', 'latin1')
    delimiters = ('\t', ',', '|')
    for encoding in encodings:
        for delimiter in delimiters:
            try:
                frame = pd.read_csv(
                    path,
                    sep=delimiter,
                    dtype=str,
                    keep_default_na=False,
                    encoding=encoding,
                    engine='python',
                )
            except UnicodeDecodeError:
                continue
            except pd.errors.ParserError:
                try:
                    frame = pd.read_csv(
                        path,
                        sep=delimiter,
                        dtype=str,
                        keep_default_na=False,
                        encoding=encoding,
                        engine='python',
                        on_bad_lines='skip',
                    )
                except Exception:
                    continue
            except Exception:
                continue

            frame.columns = [str(column).strip() for column in frame.columns]
            frame = frame.fillna('')
            if not frame.empty:
                frame = frame.iloc[1:].reset_index(drop=True)
            return frame
    return pd.DataFrame()


def load_image03_attribute_lookup(package_id: str, download_dir: Path) -> dict[str, tuple[str, str]]:
    package_dir = download_dir / f'Package_{package_id}'
    image03_candidates = [
        package_dir / 'image03.txt',
        package_dir / 'IMAGE03.txt',
    ]
    image03_path = next((path for path in image03_candidates if path.exists()), None)
    if image03_path is None:
        return {}

    frame = _read_table_with_fallback(image03_path)
    if frame.empty:
        return {}

    modality_series = frame.get('image_modality', pd.Series('', index=frame.index)).astype(str)
    description_series = frame.get('image_description', pd.Series('', index=frame.index)).astype(str)
    accession_series = frame.get('accession_number', pd.Series('', index=frame.index)).astype(str)
    dicom_ref_series = frame.get('image_file', pd.Series('', index=frame.index)).astype(str)
    jpg_ref_series = frame.get('image_thumbnail_file', pd.Series('', index=frame.index)).astype(str)

    lookup: dict[str, tuple[str, str]] = {}
    for row_idx in frame.index:
        modality = modality_series.loc[row_idx]
        description = description_series.loc[row_idx]
        image_type = _classify_image_type(modality, description)
        joint = _classify_joint_from_description(description)

        accession_id = _normalize_asset_id_token(accession_series.loc[row_idx])
        dicom_id = _normalize_asset_id_token(dicom_ref_series.loc[row_idx])
        jpg_id = _normalize_asset_id_token(jpg_ref_series.loc[row_idx])
        candidates = [accession_id, dicom_id, jpg_id]
        for asset_id in candidates:
            if asset_id and asset_id not in lookup:
                lookup[asset_id] = (image_type, joint)

    return lookup


def _infer_image_attributes_for_metadata_row(
    *,
    short_name: str | None,
    download_alias: str | None,
    nda_s3_url: str | None,
    package_file_id: str | None,
    image03_lookup: dict[str, tuple[str, str]] | None,
) -> tuple[str, str]:
    candidate_asset_id = _normalize_asset_id_token(download_alias or nda_s3_url)
    if image03_lookup is not None and candidate_asset_id:
        mapped = image03_lookup.get(candidate_asset_id)
        if mapped is not None:
            return mapped

    hint_text = ' '.join(
        value
        for value in (
            short_name,
            download_alias,
            nda_s3_url,
            package_file_id,
        )
        if value
    )
    image_type = _classify_image_type(short_name, hint_text)
    joint = _classify_joint_from_description(hint_text)
    return image_type, joint


def collect_download_filters() -> dict[str, object]:
    all_image_types = ('mri', 'x-ray', 'other')
    all_joints = ('knee', 'hip', 'hand')

    datastructure = (datastructure_text.value or '').strip().lower()
    metadata_contains = (metadata_contains_text.value or '').strip().lower()
    file_regex = (file_regex_text.value or '').strip()
    selected_extensions = tuple(
        sorted(
            {
                _infer_file_extension(str(value).strip())
                for value in extension_filter_select.value
                if str(value).strip()
            }
        )
    )
    selected_image_types = tuple(
        value
        for value in all_image_types
        if value in {str(item).strip().lower() for item in image_type_select.value}
    )
    if not selected_image_types:
        selected_image_types = all_image_types

    selected_joints = tuple(
        value
        for value in all_joints
        if value in {str(item).strip().lower() for item in joint_filter_select.value}
    )
    if not selected_joints:
        selected_joints = all_joints

    image_type_filter_active = set(selected_image_types) != set(all_image_types)
    joint_filter_active = set(selected_joints) != set(all_joints)
    use_metadata_links = bool(
        metadata_contains
        or selected_extensions
        or image_type_filter_active
        or joint_filter_active
    )
    return {
        'datastructure': datastructure,
        'metadata_contains': metadata_contains,
        'file_regex': file_regex,
        'extensions': selected_extensions,
        'image_types': selected_image_types,
        'joints': selected_joints,
        'image_type_filter_active': image_type_filter_active,
        'joint_filter_active': joint_filter_active,
        'use_metadata_links': use_metadata_links,
    }


def _format_filter_summary(filters: dict[str, object]) -> str:
    parts: list[str] = []
    datastructure = str(filters.get('datastructure') or '').strip()
    metadata_contains = str(filters.get('metadata_contains') or '').strip()
    file_regex = str(filters.get('file_regex') or '').strip()
    extensions = tuple(filters.get('extensions') or ())
    image_types = tuple(filters.get('image_types') or ())
    joints = tuple(filters.get('joints') or ())
    if datastructure:
        parts.append(f'ds={datastructure}')
    if metadata_contains:
        parts.append(f'meta~{metadata_contains}')
    if extensions:
        parts.append('ext=' + ','.join(str(value) for value in extensions))
    if image_types:
        parts.append('image=' + ','.join(str(value) for value in image_types))
    if joints:
        parts.append('joint=' + ','.join(str(value) for value in joints))
    if file_regex:
        parts.append(f'regex={file_regex}')
    return '; '.join(parts) if parts else 'none'


def build_filtered_s3_links_file(
    package_id: str,
    filters: dict[str, object],
    *,
    download_dir: Path,
    image03_lookup: dict[str, tuple[str, str]] | None = None,
) -> tuple[Path | None, int, int]:
    if not bool(filters.get('use_metadata_links', False)):
        return None, 0, 0

    metadata_path = find_package_file_metadata_path(package_id)
    if metadata_path is None:
        return None, 0, 0

    datastructure = str(filters.get('datastructure') or '').strip().lower()
    metadata_contains = str(filters.get('metadata_contains') or '').strip().lower()
    selected_extensions = {
        str(value).strip().lower()
        for value in tuple(filters.get('extensions') or ())
        if str(value).strip()
    }
    selected_image_types = {
        str(value).strip().lower()
        for value in tuple(filters.get('image_types') or ())
        if str(value).strip()
    }
    selected_joints = {
        str(value).strip().lower()
        for value in tuple(filters.get('joints') or ())
        if str(value).strip()
    }
    image_type_filter_active = bool(filters.get('image_type_filter_active', False))
    joint_filter_active = bool(filters.get('joint_filter_active', False))

    resolved_image03_lookup = image03_lookup
    if resolved_image03_lookup is None:
        resolved_image03_lookup = load_image03_attribute_lookup(package_id, download_dir)

    output_dir = package_metadata_dir(package_id) / '.download-progress' / 'filter-links'
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f'filtered-s3-links-{package_id}.txt'

    total_rows = 0
    selected_rows = 0
    unique_s3_urls: set[str] = set()

    try:
        with _open_csv_text(metadata_path) as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                total_rows += 1
                short_name = _optional_text(row.get('SHORT_NAME') or row.get('short_name'))
                download_alias = _optional_text(row.get('DOWNLOAD_ALIAS') or row.get('download_alias'))
                nda_s3_url = _optional_text(row.get('NDA_S3_URL') or row.get('nda_s3_url'))
                package_file_id = _optional_text(row.get('PACKAGE_FILE_ID') or row.get('package_file_id'))

                inferred_short_name = ''
                if short_name is not None:
                    inferred_short_name = short_name.strip().lower()
                elif download_alias is not None:
                    inferred_short_name = download_alias.split('/', maxsplit=1)[0].strip().lower()

                if datastructure and inferred_short_name != datastructure:
                    continue

                extension = _infer_file_extension(download_alias or nda_s3_url)
                if selected_extensions and extension not in selected_extensions:
                    continue

                image_type, joint = _infer_image_attributes_for_metadata_row(
                    short_name=short_name,
                    download_alias=download_alias,
                    nda_s3_url=nda_s3_url,
                    package_file_id=package_file_id,
                    image03_lookup=resolved_image03_lookup,
                )
                if image_type_filter_active and image_type not in selected_image_types:
                    continue
                if joint_filter_active and joint not in selected_joints:
                    continue

                if metadata_contains:
                    metadata_text = ' '.join(
                        value
                        for value in (
                            short_name,
                            download_alias,
                            nda_s3_url,
                            package_file_id,
                            image_type,
                            joint,
                        )
                        if value
                    ).lower()
                    if metadata_contains not in metadata_text:
                        continue

                if nda_s3_url is None:
                    continue
                if nda_s3_url in unique_s3_urls:
                    continue
                unique_s3_urls.add(nda_s3_url)
                selected_rows += 1
    except Exception:
        return None, 0, total_rows

    if selected_rows <= 0:
        try:
            if output_path.exists():
                output_path.unlink()
        except Exception:
            pass
        return None, 0, total_rows

    with output_path.open('w', encoding='utf-8', newline='\n') as handle:
        for s3_url in sorted(unique_s3_urls):
            handle.write(s3_url + '\n')

    return output_path, selected_rows, total_rows


def build_download_command(
    package_id: str,
    download_dir: Path,
    filters: dict[str, object],
    filtered_links_file: Path | None = None,
) -> list[str]:
    cmd = [
        'uvx',
        '--from',
        'nda-tools',
        'downloadcmd',
        '-dp',
        package_id,
        '-d',
        str(download_dir),
        '-wt',
        str(worker_threads.value),
    ]
    if filtered_links_file is not None:
        cmd.extend(['-t', str(filtered_links_file)])
    else:
        datastructure = str(filters.get('datastructure') or '').strip()
        if datastructure:
            cmd.extend(['-ds', datastructure])

    file_regex = str(filters.get('file_regex') or '').strip()
    if file_regex:
        cmd.extend(['--file-regex', file_regex])

    nda_user = username_text.value.strip()
    if nda_user:
        cmd.extend(['-u', nda_user])
    if verify_only.value:
        cmd.append('--verify')
    if verbose_logs.value:
        cmd.append('--verbose')
    return cmd


def _optional_text(value: str | None) -> str | None:
    text = str(value or '').strip()
    return text if text else None


def _parse_int(value: object) -> int | None:
    text = str(value or '').strip().replace(',', '')
    if not text:
        return None
    try:
        return int(text)
    except Exception:
        try:
            return int(float(text))
        except Exception:
            return None


def _infer_file_extension(path_text: str | None) -> str:
    candidate = _optional_text(path_text)
    if not candidate:
        return '[no_ext]'
    file_name = Path(candidate.split('?', maxsplit=1)[0]).name.lower()
    if not file_name:
        return '[no_ext]'
    for multi_suffix in ('.nii.gz', '.tar.gz', '.tsv.gz', '.csv.gz'):
        if file_name.endswith(multi_suffix):
            return multi_suffix
    suffix = Path(file_name).suffix.lower()
    return suffix if suffix else '[no_ext]'


def _open_csv_text(csv_path: Path):
    if csv_path.suffix.lower() == '.gz':
        import gzip

        return gzip.open(csv_path, 'rt', encoding='utf-8', newline='')
    return csv_path.open('r', encoding='utf-8', newline='')


def _format_duration(seconds: float) -> str:
    total_seconds = max(0, int(round(seconds)))
    hours, rem = divmod(total_seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f'{hours}h {minutes:02d}m {secs:02d}s'
    if minutes:
        return f'{minutes}m {secs:02d}s'
    return f'{secs}s'


def _format_eta(eta_seconds: float | None) -> str:
    if eta_seconds is None:
        return 'ETA --'
    remaining = max(0.0, eta_seconds)
    if remaining <= 0:
        return 'ETA now'
    finish_at = time.strftime('%H:%M:%S', time.localtime(time.time() + remaining))
    return f'ETA {_format_duration(remaining)} ({finish_at})'


def _format_percent(completed: int, total: int | None) -> str:
    if total is None or total <= 0:
        return '--%'
    safe_completed = min(max(0, completed), total)
    return f'{(safe_completed / total) * 100:5.1f}%'


def _format_bytes_per_second(value: float | None) -> str:
    if value is None:
        return '--'
    rate = max(0.0, float(value))
    units = ['B/s', 'KB/s', 'MB/s', 'GB/s', 'TB/s']
    unit_idx = 0
    while rate >= 1024.0 and unit_idx < len(units) - 1:
        rate /= 1024.0
        unit_idx += 1
    return f'{rate:0.2f} {units[unit_idx]}'


def set_network_usage(
    current_bytes_per_sec: float | None,
    avg_bytes_per_sec: float | None,
    force: bool = False,
) -> None:
    if not _should_emit_progress_update(
        'network_usage',
        force=force,
        interval_seconds=NETWORK_UPDATE_INTERVAL_SECONDS,
    ):
        return
    current_text = _format_bytes_per_second(current_bytes_per_sec)
    avg_text = _format_bytes_per_second(avg_bytes_per_sec)
    network_usage_html.value = f'<b>Network:</b> {current_text} (avg {avg_text})'


def set_overall_progress_meta(
    completed_packages: int,
    total_packages: int,
    eta_seconds: float | None,
    force: bool = False,
) -> None:
    if not _should_emit_progress_update('overall_progress_meta', force=force):
        return
    percent_text = _format_percent(completed_packages, total_packages)
    overall_progress_meta_html.value = (
        f'<span style=\"white-space:nowrap;\"><b>{percent_text}</b> | {_format_eta(eta_seconds)}</span>'
    )


def estimate_overall_eta_seconds(
    run_started_monotonic: float,
    completed_packages: int,
    total_packages: int,
    package_completed_files: dict[str, int],
    package_expected_files: dict[str, int | None],
) -> float | None:
    elapsed = max(0.0, time.monotonic() - run_started_monotonic)
    if elapsed <= 0:
        return None

    known_expected_values: list[int] = []
    for value in package_expected_files.values():
        if value is None:
            continue
        known_expected_values.append(max(0, int(value)))

    if known_expected_values:
        expected_fallback = sum(known_expected_values) / len(known_expected_values)
        projected_total_expected = 0.0
        projected_total_completed = 0.0
        for package_id, expected_value in package_expected_files.items():
            expected_for_package = float(expected_value) if expected_value is not None else expected_fallback
            expected_for_package = max(0.0, expected_for_package)
            projected_total_expected += expected_for_package
            completed_for_package = max(0.0, float(package_completed_files.get(package_id, 0)))
            projected_total_completed += min(completed_for_package, expected_for_package)

        if projected_total_expected <= 0:
            return 0.0
        if projected_total_completed > 0:
            file_rate = projected_total_completed / elapsed
            if file_rate > 0:
                return max(0.0, (projected_total_expected - projected_total_completed) / file_rate)

    if completed_packages <= 0:
        return None
    package_rate = completed_packages / elapsed
    if package_rate <= 0:
        return None
    return max(0.0, (total_packages - completed_packages) / package_rate)


def count_csv_data_rows(csv_path: Path | None) -> int:
    if csv_path is None or not csv_path.exists():
        return 0
    try:
        with csv_path.open('r', encoding='utf-8', newline='') as handle:
            return max(0, sum(1 for _ in handle) - 1)
    except Exception:
        return 0


def package_metadata_dir(package_id: str) -> Path:
    return Path.home() / 'NDA' / 'nda-tools' / 'downloadcmd' / 'packages' / str(package_id)


def find_package_file_metadata_path(package_id: str) -> Path | None:
    package_dir = package_metadata_dir(package_id)
    plain = package_dir / f'package_file_metadata_{package_id}.txt'
    compressed = package_dir / f'package_file_metadata_{package_id}.txt.gz'
    if plain.exists():
        return plain
    if compressed.exists():
        return compressed
    return None


def load_package_file_metadata(package_id: str) -> dict[str, object] | None:
    metadata_path = find_package_file_metadata_path(package_id)
    if metadata_path is None:
        return None

    ext_by_file_id: dict[str, str] = {}
    size_by_file_id: dict[str, int] = {}
    total_by_ext: dict[str, int] = {}
    total_bytes_by_ext: dict[str, int] = {}
    total_files = 0
    total_bytes = 0

    try:
        with _open_csv_text(metadata_path) as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                file_id = _optional_text(
                    row.get('PACKAGE_FILE_ID')
                    or row.get('package_file_id')
                    or row.get('package_fileid')
                )
                if file_id is None:
                    continue

                path_text = _optional_text(
                    row.get('DOWNLOAD_ALIAS')
                    or row.get('package_file_expected_location')
                    or row.get('NDA_S3_URL')
                    or row.get('nda_s3_url')
                    or row.get('SHORT_NAME')
                )
                extension = _infer_file_extension(path_text)
                file_size = _parse_int(
                    row.get('FILE_SIZE')
                    or row.get('expected_file_size')
                    or row.get('actual_file_size')
                ) or 0

                ext_by_file_id[file_id] = extension
                size_by_file_id[file_id] = max(0, file_size)

                total_files += 1
                total_bytes += max(0, file_size)
                total_by_ext[extension] = total_by_ext.get(extension, 0) + 1
                total_bytes_by_ext[extension] = total_bytes_by_ext.get(extension, 0) + max(0, file_size)
    except Exception:
        return None

    return {
        'metadata_path': metadata_path,
        'ext_by_file_id': ext_by_file_id,
        'size_by_file_id': size_by_file_id,
        'total_by_ext': total_by_ext,
        'total_bytes_by_ext': total_bytes_by_ext,
        'total_files': total_files,
        'total_bytes': total_bytes,
    }


def analyze_progress_report(
    report_path: Path | None,
    baseline_rows: int,
    ext_by_file_id: dict[str, str] | None = None,
    size_by_file_id: dict[str, int] | None = None,
) -> dict[str, object]:
    summary: dict[str, object] = {
        'total_rows': 0,
        'completed_this_run': 0,
        'completed_run_by_ext': {},
        'completed_run_bytes': 0,
        'completed_all_by_ext': {},
        'completed_all_bytes_by_ext': {},
    }
    if report_path is None or not report_path.exists():
        return summary

    completed_run_by_ext: dict[str, int] = summary['completed_run_by_ext']  # type: ignore[assignment]
    completed_all_by_ext: dict[str, int] = summary['completed_all_by_ext']  # type: ignore[assignment]
    completed_all_bytes_by_ext: dict[str, int] = summary['completed_all_bytes_by_ext']  # type: ignore[assignment]

    try:
        with report_path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            for row_index, row in enumerate(reader, start=1):
                total_rows = int(summary['total_rows']) + 1
                summary['total_rows'] = total_rows

                file_id = _optional_text(
                    row.get('package_file_id')
                    or row.get('PACKAGE_FILE_ID')
                )
                extension = None
                if file_id is not None and ext_by_file_id is not None:
                    extension = ext_by_file_id.get(file_id)
                if extension is None:
                    extension = _infer_file_extension(
                        _optional_text(
                            row.get('package_file_expected_location')
                            or row.get('DOWNLOAD_ALIAS')
                            or row.get('nda_s3_url')
                            or row.get('NDA_S3_URL')
                        )
                    )

                file_size = _parse_int(
                    row.get('expected_file_size')
                    or row.get('actual_file_size')
                    or row.get('FILE_SIZE')
                )
                if (file_size is None or file_size <= 0) and file_id is not None and size_by_file_id is not None:
                    file_size = size_by_file_id.get(file_id, 0)
                if file_size is None or file_size < 0:
                    file_size = 0

                completed_all_by_ext[extension] = completed_all_by_ext.get(extension, 0) + 1
                completed_all_bytes_by_ext[extension] = completed_all_bytes_by_ext.get(extension, 0) + file_size

                if row_index <= baseline_rows:
                    continue

                summary['completed_this_run'] = int(summary['completed_this_run']) + 1
                completed_run_by_ext[extension] = completed_run_by_ext.get(extension, 0) + 1
                summary['completed_run_bytes'] = int(summary['completed_run_bytes']) + file_size
    except Exception:
        return summary

    return summary


def estimate_package_eta_seconds(
    completed_this_run: int,
    expected_files_this_run: int | None,
    package_started_monotonic: float,
    report_summary: dict[str, object] | None,
    package_metadata: dict[str, object] | None,
) -> float | None:
    elapsed = max(0.0, time.monotonic() - package_started_monotonic)
    if elapsed <= 0:
        return None

    if expected_files_this_run is not None and expected_files_this_run > 0:
        if completed_this_run >= expected_files_this_run:
            return 0.0

    if report_summary is not None and package_metadata is not None:
        total_bytes_by_ext = package_metadata.get('total_bytes_by_ext')
        completed_all_bytes_by_ext = report_summary.get('completed_all_bytes_by_ext')
        completed_run_bytes = int(report_summary.get('completed_run_bytes', 0))

        if isinstance(total_bytes_by_ext, dict) and isinstance(completed_all_bytes_by_ext, dict):
            remaining_bytes = 0
            for ext, total_bytes in total_bytes_by_ext.items():
                done_bytes = _parse_int(completed_all_bytes_by_ext.get(ext)) or 0
                remaining_bytes += max(0, int(total_bytes) - done_bytes)
            if remaining_bytes <= 0:
                return 0.0
            if completed_run_bytes > 0:
                byte_rate = completed_run_bytes / elapsed
                if byte_rate > 0:
                    return max(0.0, remaining_bytes / byte_rate)

    if completed_this_run > 0 and expected_files_this_run is not None and expected_files_this_run >= completed_this_run:
        file_rate = completed_this_run / elapsed
        if file_rate > 0:
            return max(0.0, (expected_files_this_run - completed_this_run) / file_rate)

    return None


def find_download_progress_report(
    package_id: str,
    download_dir: Path,
    filters: dict[str, object] | None = None,
    filtered_links_file: Path | None = None,
) -> Path | None:
    package_meta_dir = package_metadata_dir(package_id)
    progress_root = package_meta_dir / '.download-progress'
    manifest_path = progress_root / 'download-job-manifest.csv'
    if not manifest_path.exists():
        return None

    target_dir = download_dir.expanduser().resolve()
    try:
        target_package_id = int(package_id)
    except ValueError:
        return None

    expected_datastructure = ''
    expected_regex = ''
    if filters is not None:
        expected_datastructure = str(filters.get('datastructure') or '').strip().lower()
        expected_regex = str(filters.get('file_regex') or '').strip()

    expected_links_file = None
    if filtered_links_file is not None:
        try:
            expected_links_file = filtered_links_file.expanduser().resolve()
        except Exception:
            expected_links_file = filtered_links_file

    matched_report: Path | None = None
    try:
        with manifest_path.open('r', encoding='utf-8', newline='') as handle:
            for row in csv.DictReader(handle):
                try:
                    row_package_id = int(str(row.get('package_id', '')).strip())
                except Exception:
                    continue
                if row_package_id != target_package_id:
                    continue

                row_dir_text = _optional_text(row.get('download_directory'))
                if row_dir_text is None:
                    continue
                try:
                    row_dir = Path(row_dir_text).expanduser().resolve()
                except Exception:
                    continue
                if row_dir != target_dir:
                    continue

                if _optional_text(row.get('s3_destination')) is not None:
                    continue

                row_ds = (_optional_text(row.get('data_structure')) or '').strip().lower()
                row_regex = _optional_text(row.get('regex')) or ''
                row_links = _optional_text(row.get('s3_links_file'))

                if expected_links_file is not None:
                    if row_links is None:
                        continue
                    try:
                        row_links_path = Path(row_links).expanduser().resolve()
                    except Exception:
                        continue
                    if row_links_path != expected_links_file:
                        continue
                else:
                    if row_links is not None:
                        continue
                    if expected_datastructure:
                        if row_ds != expected_datastructure:
                            continue
                    elif row_ds:
                        continue

                if expected_regex:
                    if row_regex != expected_regex:
                        continue
                elif row_regex:
                    continue

                job_uuid = _optional_text(row.get('uuid'))
                if not job_uuid:
                    continue
                matched_report = progress_root / job_uuid / 'download-progress-report.csv'
    except Exception:
        return None

    return matched_report


def parse_expected_total_from_line(line: str, current_expected: int | None) -> int | None:
    expected = current_expected

    begin_match = re.search(r'Beginning download of (?:the remaining )?(\d+) files', line)
    if begin_match:
        expected = int(begin_match.group(1))

    queue_match = re.search(r'Queue contains (\d+) files', line)
    if queue_match:
        queued_total = int(queue_match.group(1))
        expected = max(expected or 0, queued_total)

    report_match = re.search(r'(\d+)/(\d+) queued files downloaded so far', line)
    if report_match:
        reported_total = int(report_match.group(2))
        expected = max(expected or 0, reported_total)

    return expected


def set_package_file_progress(
    package_id: str,
    completed: int,
    expected: int | None,
    eta_seconds: float | None = None,
    force: bool = False,
) -> None:
    if not _should_emit_progress_update(f'package_file_progress:{package_id}', force=force):
        return

    completed = max(0, int(completed))
    if expected is not None and expected >= 0:
        expected_int = max(0, int(expected))
        if expected_int == 0:
            package_file_progress.max = 1
            package_file_progress.value = 1
            package_files_html.value = f'<b>Package {package_id} files:</b> 0 / 0'
            percent_text = '100.0%'
        else:
            package_file_progress.max = expected_int
            package_file_progress.value = min(completed, expected_int)
            package_files_html.value = f'<b>Package {package_id} files:</b> {package_file_progress.value} / {expected_int}'
            percent_text = _format_percent(package_file_progress.value, expected_int)
    else:
        package_file_progress.max = max(1, completed)
        package_file_progress.value = min(completed, package_file_progress.max)
        package_files_html.value = f'<b>Package {package_id} files:</b> {completed} / ?'
        percent_text = _format_percent(completed, None)

    package_file_meta_html.value = (
        f'<span style=\"white-space:nowrap;\"><b>{percent_text}</b> | {_format_eta(eta_seconds)}</span>'
    )


def monitor_download_progress(
    package_id: str,
    download_dir: Path,
    download_filters: dict[str, object],
    filtered_links_file: Path | None,
    baseline_rows: int,
    expected_ref: dict[str, int | None],
    stop_event: threading.Event,
    initial_report_path: Path | None,
    package_started_monotonic: float,
    package_metadata: dict[str, object] | None = None,
    on_progress=None,
) -> None:
    report_path = initial_report_path
    ext_by_file_id = None
    size_by_file_id = None
    last_rate_snapshot = {
        'time': package_started_monotonic,
        'bytes': 0,
    }
    if package_metadata is not None:
        ext_by_file_id = package_metadata.get('ext_by_file_id')
        size_by_file_id = package_metadata.get('size_by_file_id')

    def publish(summary: dict[str, object]) -> None:
        completed_this_run = int(summary.get('completed_this_run', 0))
        expected_value = expected_ref.get('value')
        completed_run_bytes = max(0, int(summary.get('completed_run_bytes', 0)))
        now_monotonic = time.monotonic()
        elapsed = max(0.0, now_monotonic - package_started_monotonic)
        avg_bytes_per_sec = (completed_run_bytes / elapsed) if elapsed > 0 else None
        prior_time = float(last_rate_snapshot.get('time', now_monotonic))
        prior_bytes = int(last_rate_snapshot.get('bytes', completed_run_bytes))
        delta_time = max(1e-9, now_monotonic - prior_time)
        delta_bytes = max(0, completed_run_bytes - prior_bytes)
        current_bytes_per_sec = delta_bytes / delta_time
        last_rate_snapshot['time'] = now_monotonic
        last_rate_snapshot['bytes'] = completed_run_bytes

        eta_seconds = estimate_package_eta_seconds(
            completed_this_run=completed_this_run,
            expected_files_this_run=expected_value,
            package_started_monotonic=package_started_monotonic,
            report_summary=summary,
            package_metadata=package_metadata,
        )
        if on_progress is not None:
            on_progress(completed_this_run, expected_value, eta_seconds, current_bytes_per_sec, avg_bytes_per_sec)
        else:
            set_package_file_progress(package_id, completed_this_run, expected_value, eta_seconds)
            set_network_usage(current_bytes_per_sec, avg_bytes_per_sec)

    while not stop_event.is_set():
        if report_path is None or not report_path.exists():
            report_path = find_download_progress_report(
                package_id,
                download_dir,
                filters=download_filters,
                filtered_links_file=filtered_links_file,
            )
        summary = analyze_progress_report(
            report_path=report_path,
            baseline_rows=baseline_rows,
            ext_by_file_id=ext_by_file_id if isinstance(ext_by_file_id, dict) else None,
            size_by_file_id=size_by_file_id if isinstance(size_by_file_id, dict) else None,
        )
        publish(summary)
        time.sleep(MONITOR_SCAN_INTERVAL_SECONDS)

    if report_path is None or not report_path.exists():
        report_path = find_download_progress_report(
            package_id,
            download_dir,
            filters=download_filters,
            filtered_links_file=filtered_links_file,
        )
    summary = analyze_progress_report(
        report_path=report_path,
        baseline_rows=baseline_rows,
        ext_by_file_id=ext_by_file_id if isinstance(ext_by_file_id, dict) else None,
        size_by_file_id=size_by_file_id if isinstance(size_by_file_id, dict) else None,
    )
    publish(summary)


def run_and_stream_output(cmd: list[str], on_line=None) -> int:
    process = subprocess.Popen(
        cmd,
        cwd=repo_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    with runtime_lock:
        runtime_state['active_process'] = process

    try:
        assert process.stdout is not None
        for line in process.stdout:
            stripped = line.rstrip()
            append_log(stripped)
            if line_has_dns_resolution_error(stripped):
                trigger_auto_pause_due_to_network(stripped)
            if on_line is not None:
                on_line(stripped)
        process.wait()
        return int(process.returncode)
    finally:
        with runtime_lock:
            if runtime_state.get('active_process') is process:
                runtime_state['active_process'] = None


def run_downloads_worker(
    package_ids: list[str],
    download_dir: Path,
    download_filters: dict[str, object],
) -> None:
    failures: list[str] = []
    paused = False
    canceled = False
    keep_auto_pause_state = False
    package_metadata_cache: dict[str, dict[str, object] | None] = {}
    image03_attribute_cache: dict[str, dict[str, tuple[str, str]]] = {}
    package_completed_files: dict[str, int] = {package_id: 0 for package_id in package_ids}
    package_expected_files: dict[str, int | None] = {package_id: None for package_id in package_ids}
    run_started_monotonic = time.monotonic()

    def refresh_overall_meta(completed_packages: int, force: bool = False) -> None:
        eta_seconds = estimate_overall_eta_seconds(
            run_started_monotonic=run_started_monotonic,
            completed_packages=completed_packages,
            total_packages=len(package_ids),
            package_completed_files=package_completed_files,
            package_expected_files=package_expected_files,
        )
        set_overall_progress_meta(completed_packages, len(package_ids), eta_seconds, force=force)

    try:
        with runtime_lock:
            progress_update_state.clear()
        output_area.clear_output(wait=True)
        overall_progress.max = len(package_ids)
        overall_progress.value = 0
        overall_progress.bar_style = ''
        package_file_progress.bar_style = ''
        set_overall_progress_meta(0, len(package_ids), None, force=True)
        package_file_meta_html.value = '<span style="white-space:nowrap;">0.0% | ETA --</span>'
        set_network_usage(None, None, force=True)

        append_log(f'Download target directory: {download_dir}')
        append_log(f'Package count: {len(package_ids)}')
        append_log(f'Filters: {_format_filter_summary(download_filters)}')
        append_log('---')

        for idx, package_id in enumerate(package_ids, start=1):
            if is_cancel_requested():
                canceled = True
                append_log('Cancel requested. Ending before next package starts.')
                break
            if is_pause_requested():
                paused = True
                append_log('Pause requested. Ending before next package starts.')
                break

            current_package_html.value = (
                f'<b>Current package:</b> {package_id} '
                f'({idx}/{len(package_ids)})'
            )
            completed_packages_before_current = idx - 1

            if package_id not in package_metadata_cache:
                package_metadata_cache[package_id] = load_package_file_metadata(package_id)
            package_metadata = package_metadata_cache.get(package_id)

            filtered_links_file: Path | None = None
            if bool(download_filters.get('use_metadata_links', False)):
                if package_id not in image03_attribute_cache:
                    image03_attribute_cache[package_id] = load_image03_attribute_lookup(package_id, download_dir)
                image03_lookup = image03_attribute_cache.get(package_id, {})
                filtered_links_file, selected_rows, total_rows = build_filtered_s3_links_file(
                    package_id,
                    download_filters,
                    download_dir=download_dir,
                    image03_lookup=image03_lookup,
                )
                if filtered_links_file is None:
                    if total_rows <= 0:
                        append_log(
                            f'[{idx}/{len(package_ids)}] Package {package_id}: '
                            'metadata index unavailable, proceeding without metadata-link filtering.'
                        )
                    else:
                        append_log(
                            f'[{idx}/{len(package_ids)}] Package {package_id}: '
                            f'no metadata-filter matches ({selected_rows}/{total_rows}). Skipping package.'
                        )
                        set_package_file_progress(package_id, completed=0, expected=0, eta_seconds=0.0, force=True)
                        set_network_usage(0.0, 0.0, force=True)
                        overall_progress.value = idx
                        refresh_overall_meta(idx, force=True)
                        append_log('---')
                        continue
                else:
                    append_log(
                        f'[{idx}/{len(package_ids)}] Package {package_id}: '
                        f'metadata filter selected {selected_rows}/{total_rows} files.'
                    )

            cmd = build_download_command(
                package_id=package_id,
                download_dir=download_dir,
                filters=download_filters,
                filtered_links_file=filtered_links_file,
            )
            append_log(f'[{idx}/{len(package_ids)}] Package {package_id}')
            append_log('CMD: ' + ' '.join(shlex.quote(part) for part in cmd))

            expected_ref: dict[str, int | None] = {'value': package_expected_files.get(package_id)}
            progress_report_path = find_download_progress_report(
                package_id,
                download_dir,
                filters=download_filters,
                filtered_links_file=filtered_links_file,
            )
            baseline_rows = count_csv_data_rows(progress_report_path)
            if package_metadata is not None:
                total_files = _parse_int(package_metadata.get('total_files')) or 0
                expected_ref['value'] = max(0, total_files - baseline_rows)
                package_expected_files[package_id] = expected_ref['value']

            package_started_monotonic = time.monotonic()
            latest_eta_ref: dict[str, float | None] = {'value': None}
            latest_network_ref: dict[str, float | None] = {
                'current': None,
                'avg': None,
            }

            def publish_package_progress(
                completed: int,
                expected: int | None,
                eta_seconds: float | None,
                current_bytes_per_sec: float | None = None,
                avg_bytes_per_sec: float | None = None,
                force: bool = False,
            ) -> None:
                completed_int = max(0, int(completed))
                expected_int = max(0, int(expected)) if expected is not None else None
                latest_eta_ref['value'] = eta_seconds
                if current_bytes_per_sec is not None:
                    latest_network_ref['current'] = current_bytes_per_sec
                if avg_bytes_per_sec is not None:
                    latest_network_ref['avg'] = avg_bytes_per_sec
                package_completed_files[package_id] = completed_int
                if expected_int is not None:
                    package_expected_files[package_id] = expected_int
                set_package_file_progress(package_id, completed_int, expected_int, eta_seconds, force=force)
                set_network_usage(latest_network_ref.get('current'), latest_network_ref.get('avg'), force=force)
                refresh_overall_meta(completed_packages_before_current, force=force)

            publish_package_progress(completed=0, expected=expected_ref['value'], eta_seconds=None, force=True)

            monitor_stop_event: threading.Event | None = None
            monitor_thread: threading.Thread | None = None

            if not verify_only.value:
                monitor_stop_event = threading.Event()
                monitor_thread = threading.Thread(
                    target=monitor_download_progress,
                    args=(
                        package_id,
                        download_dir,
                        download_filters,
                        filtered_links_file,
                        baseline_rows,
                        expected_ref,
                        monitor_stop_event,
                        progress_report_path,
                        package_started_monotonic,
                        package_metadata,
                        publish_package_progress,
                    ),
                    daemon=True,
                )
                monitor_thread.start()

            def handle_log_line(line: str) -> None:
                prior_expected = expected_ref['value']
                expected_ref['value'] = parse_expected_total_from_line(line, expected_ref['value'])
                if expected_ref['value'] is not None:
                    expected_ref['value'] = max(0, int(expected_ref['value']))
                    package_expected_files[package_id] = expected_ref['value']
                report_match = re.search(r'(\d+)/(\d+) queued files downloaded so far', line)
                if report_match:
                    from_log_completed = int(report_match.group(1))
                    from_log_total = int(report_match.group(2))
                    expected_ref['value'] = max(expected_ref['value'] or 0, from_log_total)
                    package_expected_files[package_id] = expected_ref['value']
                    publish_package_progress(
                        from_log_completed,
                        expected_ref['value'],
                        latest_eta_ref.get('value'),
                    )
                elif expected_ref['value'] != prior_expected:
                    publish_package_progress(
                        package_completed_files.get(package_id, 0),
                        expected_ref['value'],
                        latest_eta_ref.get('value'),
                    )

            try:
                return_code = run_and_stream_output(cmd, on_line=handle_log_line)
            finally:
                if monitor_stop_event is not None:
                    monitor_stop_event.set()
                if monitor_thread is not None:
                    monitor_thread.join(timeout=2.0)

            final_report_path = progress_report_path
            if final_report_path is None or not final_report_path.exists():
                final_report_path = find_download_progress_report(
                    package_id,
                    download_dir,
                    filters=download_filters,
                    filtered_links_file=filtered_links_file,
                )
            ext_by_file_id = None
            size_by_file_id = None
            if package_metadata is not None:
                ext_by_file_id = package_metadata.get('ext_by_file_id')
                size_by_file_id = package_metadata.get('size_by_file_id')
            final_summary = analyze_progress_report(
                report_path=final_report_path,
                baseline_rows=baseline_rows,
                ext_by_file_id=ext_by_file_id if isinstance(ext_by_file_id, dict) else None,
                size_by_file_id=size_by_file_id if isinstance(size_by_file_id, dict) else None,
            )
            completed_this_run = int(final_summary.get('completed_this_run', 0))
            final_eta = 0.0 if return_code == 0 else estimate_package_eta_seconds(
                completed_this_run=completed_this_run,
                expected_files_this_run=expected_ref['value'],
                package_started_monotonic=package_started_monotonic,
                report_summary=final_summary,
                package_metadata=package_metadata,
            )
            publish_package_progress(completed_this_run, expected_ref['value'], final_eta, force=True)

            expected_text = str(expected_ref['value']) if expected_ref['value'] is not None else '?'
            append_log(f'File progress summary for package {package_id}: {completed_this_run}/{expected_text}')

            if return_code != 0:
                if is_cancel_requested():
                    canceled = True
                    append_log(f'Package {package_id} canceled by user request.')
                    overall_progress.value = idx
                    refresh_overall_meta(idx, force=True)
                    append_log('---')
                    break
                if is_pause_requested():
                    paused = True
                    append_log(f'Package {package_id} paused by user request.')
                    overall_progress.value = idx
                    refresh_overall_meta(idx, force=True)
                    append_log('---')
                    break
                failures.append(package_id)
                append_log(f'Package {package_id} failed with code {return_code}.')
            else:
                append_log(f'Package {package_id} completed.')

            overall_progress.value = idx
            refresh_overall_meta(idx, force=True)
            append_log('---')

        current_package_html.value = '<b>Current package:</b> idle'
        if canceled:
            stop_auto_resume_timer()
            clear_resume_payload()
            overall_progress.bar_style = 'warning'
            package_file_progress.bar_style = 'warning'
            status_html.value = '<b style="color:#8a6d3b;">Download canceled by user.</b>'
            set_overall_progress_meta(int(overall_progress.value), len(package_ids), None, force=True)
            set_network_usage(None, None, force=True)
        elif paused:
            overall_progress.bar_style = 'warning'
            package_file_progress.bar_style = 'warning'
            with runtime_lock:
                auto_pause_in_effect = bool(runtime_state.get('auto_pause_detected', False))
            if auto_pause_in_effect:
                keep_auto_pause_state = True
                status_html.value = '<b style="color:#8a6d3b;">Download auto-paused due to DNS resolution errors. Auto-resume will retry every 1 minute.</b>'
                append_log('Auto-resume armed: retrying every 1 minute until DNS resolution recovers.')
                start_auto_resume_timer()
            else:
                status_html.value = '<b style="color:#8a6d3b;">Download paused. Click Run downloads to resume from existing files.</b>'
                stop_auto_resume_timer()
            set_overall_progress_meta(int(overall_progress.value), len(package_ids), None, force=True)
            set_network_usage(None, None, force=True)
        elif failures:
            stop_auto_resume_timer()
            failed = ', '.join(failures)
            overall_progress.bar_style = 'danger'
            package_file_progress.bar_style = 'danger'
            status_html.value = f'<b style="color:#a94442;">Finished with failures for package IDs: {failed}</b>'
            refresh_overall_meta(int(overall_progress.value), force=True)
            set_network_usage(None, None, force=True)
        else:
            stop_auto_resume_timer()
            clear_resume_payload()
            overall_progress.bar_style = 'success'
            package_file_progress.bar_style = 'success'
            status_html.value = '<b style="color:#2f6f3e;">All package commands completed successfully.</b>'
            set_overall_progress_meta(len(package_ids), len(package_ids), 0.0, force=True)
            set_network_usage(0.0, 0.0, force=True)
    except Exception as exc:
        stop_auto_resume_timer()
        overall_progress.bar_style = 'danger'
        package_file_progress.bar_style = 'danger'
        status_html.value = f'<b style="color:#a94442;">Unexpected error: {exc}</b>'
        append_log(f'Unexpected error: {exc}')
    finally:
        with runtime_lock:
            runtime_state['is_running'] = False
            runtime_state['pause_requested'] = False
            runtime_state['cancel_requested'] = False
            runtime_state['active_process'] = None
            runtime_state['worker_thread'] = None
            runtime_state['auto_pause_detected'] = keep_auto_pause_state
        set_controls_running(False)


def run_downloads(_: widgets.Button) -> None:
    selected_ids = [str(value).strip() for value in package_selector.value]
    manual_ids = parse_manual_package_ids(manual_ids_text.value)
    package_ids = list(dict.fromkeys([*selected_ids, *manual_ids]))

    if not package_ids:
        status_html.value = '<b style="color:#a94442;">No package IDs selected.</b>'
        return

    download_dir = Path(download_dir_text.value).expanduser().resolve()
    download_dir.mkdir(parents=True, exist_ok=True)
    download_filters = collect_download_filters()

    with output_area:
        clear_output(wait=True)

    state['last_runner_cmd'] = ''
    copy_runner_cmd_button.disabled = True

    if auto_install_downloader.value and not bool(state.get('downloader_env_ready', False)):
        if not ensure_downloader_environment(verbose=True):
            status_html.value = '<b style="color:#a94442;">Downloader install failed. Fix install output and try again.</b>'
            _sync_log_scroll_position()
            return
        append_log('---')

    append_log(f'Selected package IDs ({len(package_ids)}): {", ".join(package_ids)}')
    append_log(f'Download target directory: {download_dir}')
    append_log(f'Filters: {_format_filter_summary(download_filters)}')

    missing_metadata: list[str] = []
    plan_packages: list[dict[str, object]] = []

    package_timepoint_by_id: dict[str, str] = {}
    for option in state.get('all_options', []):
        if not isinstance(option, tuple) or len(option) < 2:
            continue
        label = str(option[0])
        package_token = str(option[1]).strip()
        if not package_token:
            continue
        match = re.search(r'timepoint\s*=\s*([^|]+)', label, flags=re.IGNORECASE)
        if match:
            package_timepoint_by_id[package_token] = match.group(1).strip()

    for package_id in package_ids:
        append_log('---')
        append_log(f'Package {package_id}')

        metadata_path = find_package_file_metadata_path(package_id)
        image03_lookup = None
        if bool(download_filters.get('use_metadata_links', False)):
            image03_lookup = load_image03_attribute_lookup(package_id, download_dir)

        filtered_links_file, selected_rows, total_rows = build_filtered_s3_links_file(
            package_id,
            download_filters,
            download_dir=download_dir,
            image03_lookup=image03_lookup,
        )

        if metadata_path is not None:
            append_log(f'Metadata cache: {metadata_path}')
        else:
            append_log('Metadata cache: not found under ~/.NDA metadata cache.')

        expected_total: int | None = None
        if bool(download_filters.get('use_metadata_links', False)):
            if filtered_links_file is not None:
                expected_total = int(selected_rows)
                total_display = total_rows if total_rows > 0 else selected_rows
                append_log(f'Filtered links file: {filtered_links_file} ({selected_rows}/{total_display} rows)')
            elif metadata_path is None:
                missing_metadata.append(package_id)
                append_log('Filtered links file: not generated (metadata cache missing).')
            else:
                append_log(f'Filtered links file: not generated (0 rows matched out of {total_rows}).')
        else:
            append_log('Filtered links file: not required for current filter settings.')

        cmd = build_download_command(
            package_id,
            download_dir=download_dir,
            filters=download_filters,
            filtered_links_file=filtered_links_file,
        )
        cmd_text = _cmd_join([str(part) for part in cmd])
        append_log('Terminal command (CMD):')
        append_log(cmd_text)

        plan_packages.append(
            {
                'package_id': package_id,
                'timepoint_label': package_timepoint_by_id.get(package_id, ''),
                'cmd': [str(part) for part in cmd],
                'expected_total': int(expected_total) if expected_total is not None else None,
                'filtered_links_file': str(filtered_links_file) if filtered_links_file is not None else None,
            }
        )

    plan_dir = (oai_root / 'logs' / 'oai_download_plans').resolve()
    plan_dir.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime('%Y%m%d-%H%M%S')
    plan_path = plan_dir / f'download-plan-{timestamp}.json'

    plan_payload = {
        'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'download_dir': str(download_dir),
        'filters': dict(download_filters),
        'packages': plan_packages,
    }
    plan_path.write_text(json.dumps(plan_payload, indent=2), encoding='utf-8')

    runner_cmd_parts = [
        'uv',
        'run',
        '--project',
        str(oai_root),
        '--with',
        'tqdm',
        'python',
        '-m',
        'tmc_oai.oai_download_tqdm_runner',
        '--plan',
        str(plan_path),
        '--poll-interval',
        '5',
    ]
    runner_cmd = _cmd_join(runner_cmd_parts)

    append_log('---')
    append_log(f'Plan file written: {plan_path}')
    install_cmd = str(state.get('last_install_cmd') or _cmd_join(['uv', 'sync', '--project', str(oai_root)]))
    append_log('Install/update command (CMD):')
    append_log(install_cmd)
    append_log('TQDM runner command (CMD):')
    append_log(runner_cmd)
    state['last_runner_cmd'] = runner_cmd
    copy_runner_cmd_button.disabled = False

    if missing_metadata:
        example_id = missing_metadata[0]
        prefetch_cmd = _cmd_join([
            'uvx', '--from', 'nda-tools', 'downloadcmd',
            '-dp', example_id, '-d', str(download_dir), '--verify'
        ])
        append_log('---')
        append_log('Metadata cache missing for package(s): ' + ', '.join(missing_metadata))
        append_log('Run this once per missing package to populate metadata cache:')
        append_log(prefetch_cmd)
        status_html.value = '<b style="color:#8a6d3b;">Commands and plan generated. Some metadata caches are missing; see output.</b>'
    else:
        status_html.value = '<b style="color:#2f6f3e;">Commands and tqdm runner plan generated. Copy the runner command from output.</b>'

    _sync_log_scroll_position()


def pause_current_download(_: widgets.Button) -> None:
    with runtime_lock:
        runtime_state['pause_requested'] = True
        runtime_state['auto_pause_detected'] = False
        active_process = runtime_state.get('active_process')

    stop_auto_resume_timer()
    status_html.value = '<b style="color:#8a6d3b;">Pause requested. Waiting for current package process to exit...</b>'
    current_package_html.value = '<b>Current package:</b> pause requested'
    append_log('Pause requested by user.')

    terminate_process_tree(active_process)


def cancel_current_download(_: widgets.Button) -> None:
    with runtime_lock:
        runtime_state['cancel_requested'] = True
        runtime_state['auto_pause_detected'] = False
        active_process = runtime_state.get('active_process')

    stop_auto_resume_timer()
    clear_resume_payload()
    status_html.value = '<b style="color:#8a6d3b;">Cancel requested. Waiting for current package process to exit...</b>'
    current_package_html.value = '<b>Current package:</b> cancel requested'
    append_log('Cancel requested by user.')

    terminate_process_tree(active_process)


reload_packages_button.on_click(refresh_package_options)
select_all_button.on_click(on_select_all)
clear_selection_button.on_click(on_clear_selection)
browse_button.on_click(choose_folder)
run_button.on_click(run_downloads)
copy_runner_cmd_button.on_click(copy_runner_command)
install_downloader_button.on_click(install_downloader)
filter_text.observe(on_filter_change, names='value')

control_button_row = widgets.HBox(
    [run_button, copy_runner_cmd_button, install_downloader_button],
    layout=widgets.Layout(z_index='10', position='relative', margin='0 0 6px 0'),
)
overall_progress_row = widgets.HBox([overall_progress, overall_progress_meta_html])

ui = widgets.VBox([
    widgets.HBox([map_csv_text, reload_packages_button]),
    filter_text,
    package_selector,
    widgets.HBox([select_all_button, clear_selection_button]),
    manual_ids_text,
    widgets.HBox([download_dir_text, browse_button]),
    widgets.HBox([username_text, worker_threads, verify_only, verbose_logs, auto_install_downloader]),
    widgets.HBox([image_type_select, joint_filter_select, extension_filter_select]),
    widgets.HBox([datastructure_text, file_regex_text]),
    widgets.HBox([metadata_contains_text]),
    control_button_row,
    status_html,
    output_area,
])

display(ui)
_sync_log_scroll_position()
refresh_package_options()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>